In [11]:
from helpers import coherent_state, gaussian_displacement, cat_ideal, compute_beta_norm, fidelity, catability,catability_norm
import numpy as np
import numba as na

In [12]:
def kerr_norm_catable_operator (dim, parity, alpha, gamma, beta, norm):
    '''
    Computes the operator (2) in Fock basis.

    Parameters
    ----------
    dim : integer
        Dimension of the Hilbert space restriction to construct the operator on.

    Parameters (operator)
    ---------------------
    parity : np.integer (values -1, 1)
        Parity of the target Schrodinger state.
    alpha : np.complex
        Amplitude of the target Schrodinger state.
    gamma : np.floating
        Sensitivity of the non-Gaussian fringe detection.

    Returns
    -------
    np.ndarray
        The operator matrix.
    '''

    I = np.eye(dim)
    P = np.diag((- 1.0) ** np.arange(dim, dtype = np.float64))
    A = np.diag(np.sqrt(np.arange(1, dim, dtype = np.float64)), 1)
    D= gaussian_displacement(dim,beta)
    B = A @ A
    DPD = D @ P @ np.conjugate(D).T
    u = B.T - I * (np.conj(alpha) ** 2) 
    v = (B   - I * (alpha ** 2))
    w = gamma * (np.abs(norm)*I -parity * DPD) 

    return u @ v + w

In [13]:
def kerr_catable_operator (dim, parity, alpha, gamma, beta):
    '''
    Computes the operator (2) in Fock basis.

    Parameters
    ----------
    dim : integer
        Dimension of the Hilbert space restriction to construct the operator on.

    Parameters (operator)
    ---------------------
    parity : np.integer (values -1, 1)
        Parity of the target Schrodinger state.
    alpha : np.complex
        Amplitude of the target Schrodinger state.
    gamma : np.floating
        Sensitivity of the non-Gaussian fringe detection.

    Returns
    -------
    np.ndarray
        The operator matrix.
    '''

    I = np.eye(dim)
    P = np.diag((- 1.0) ** np.arange(dim, dtype = np.float64))
    A = np.diag(np.sqrt(np.arange(1, dim, dtype = np.float64)), 1)
    D = gaussian_displacement(dim,beta)
    B = A @ A
    DPD = D @ P @ np.conjugate(D).T
    
    u = B.T - I * (np.conj(alpha) ** 2) 
    v = (B   - I * (alpha ** 2))
    w = gamma * (1*I -parity * DPD) 

    return u @ v + w

In [14]:
def cat_state_from_nullifier (dim,parity,alpha,gamma,theta):
    alpha_real = alpha.imag
    beta,_ = compute_beta_norm(alpha_real,parity,theta)
    beta = beta*parity
    op = kerr_catable_operator(dim, parity, alpha, gamma, beta)

    
    eigenval, eigenvect = np.linalg.eig(op)
    min_index = np.argmin(eigenval)
    min_eigenval = eigenval[min_index]
    cat_state = eigenvect[:,min_index]
    
    rho = np.outer(cat_state,np.conj(cat_state))
    
    return(rho,op,beta,min_eigenval)

In [15]:
def cat_state_from_nullifier_norm (dim,parity,alpha,gamma,theta):
    alpha_real = alpha.imag
    beta,norm = compute_beta_norm(alpha_real,parity,theta)
    beta = beta*parity
    op = kerr_norm_catable_operator(dim, parity, alpha, gamma, beta,norm)

    eigenval, eigenvect = np.linalg.eig(op)
    min_index = np.argmin(eigenval)
    min_eigenval = eigenval[min_index]
    cat_state = eigenvect[:,min_index]
    
    rho = np.outer(cat_state,np.conj(cat_state))
    
    return(rho,op,beta)

In [ ]:
#benchmarking the approximation
alphaspace = np.arange(0.5,4.1,0.01)
alphaspace = alphaspace *1j
np.save('results/approx/alphaspace.npy', alphaspace)
dim = 90
parity = 1
gammaspace = np.array([0.1,1.0,2.0])
np.save('results/approx/gammaspace.npy', gammaspace)
theta = np.pi/2 
for gamma in gammaspace:
    list_catability = []
    list_fidelity = []
    for alpha in alphaspace:
        rho_cat,op,_,_ = cat_state_from_nullifier(dim,parity,alpha,gamma,theta)
        rho_ideal = cat_ideal(dim, alpha,theta)
        catabil = catability(op,rho_ideal)
        fid = fidelity(rho_ideal,rho_cat)
        list_catability.append(catabil)
        list_fidelity.append(fid)
    np.save(f'results/approx/parity_plus/fidelity_{gamma}.npy', list_fidelity)
    np.save(f'results/approx/parity_plus/catability_{gamma}.npy', list_catability)

In [9]:
#benchmarking the approximation
alphaspace = np.arange(0.5,4.1,0.01)
alphaspace = alphaspace *1j
np.save('results/approx/alphaspace.npy', alphaspace)
dim = 90
parity = -1
gammaspace = np.array([0.1,1.0,2.0])
np.save('results/approx/gammaspace.npy', gammaspace)
theta = np.pi/2 
for gamma in gammaspace:
    list_catability = []
    list_fidelity = []
    for alpha in alphaspace:
        rho_cat,op,_,_ = cat_state_from_nullifier(dim,parity,alpha,gamma,theta)
        rho_ideal = cat_ideal(dim, alpha,theta)
        catabil = catability(op,rho_ideal)
        fid = fidelity(rho_ideal,rho_cat)
        list_catability.append(catabil)
        list_fidelity.append(fid)
    np.save(f'results/approx/parity_minus/fidelity_{gamma}.npy', list_fidelity)
    np.save(f'results/approx/parity_minus/catability_{gamma}.npy', list_catability)

In [18]:
#benchmarking the approximation
#parity +1
alphaspace = np.arange(0.5,4.1,0.01)
alphaspace = alphaspace *1j
np.save('results/approx/alphaspace.npy', alphaspace)
dim = 90
parity = 1
gammaspace = np.array([0.1,1.0,2.0])
np.save('results/approx/gammaspace.npy', gammaspace)
theta = np.pi/2 
for gamma in gammaspace:
    list_catability = []
    for alpha in alphaspace:
        rho_cat,op,_ = cat_state_from_nullifier_norm(dim,parity,alpha,gamma,theta)
        rho_ideal = cat_ideal(dim, alpha,theta)
        catabil = catability(op,rho_ideal)
        list_catability.append(catabil)
    np.save(f'results/approx/parity_plus/catability_{gamma}_norm.npy', list_catability)

In [17]:
#benchmarking the approximation 
#parity -1
alphaspace = np.arange(0.5,4.1,0.01)
alphaspace = alphaspace *1j
np.save('results/approx/alphaspace.npy', alphaspace)
dim = 90
parity = -1
gammaspace = np.array([0.1,1.0,2.0])
np.save('results/approx/gammaspace.npy', gammaspace)
theta = np.pi/2 
for gamma in gammaspace:
    list_catability = []
    for alpha in alphaspace:
        rho_cat,op,_ = cat_state_from_nullifier_norm(dim,parity,alpha,gamma,theta)
        rho_ideal = cat_ideal(dim, alpha,theta)
        catabil = catability(op,rho_ideal)
        list_catability.append(catabil)
    np.save(f'results/approx/parity_minus/catability_{gamma}_norm.npy', list_catability)